In [ ]:
%run ./1_config
import time

class SetupHelper:
    def __init__(self):
        self.c = conf

    def create_tables(self):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.c.lakehouse}.{self.c.schema}")
        T = self.c.table_fqn
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_listings)} (
          property_id STRING, property_type STRING, bedrooms INT, bathrooms INT,
          suburb STRING, postcode STRING, state STRING, price INT, price_range STRING,
          is_luxury BOOLEAN, land_size_sqm INT, _ingested_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_property_views)} (
          event_id STRING, session_id STRING, event_ts TIMESTAMP, event_type STRING,
          user_id STRING, ip STRING, user_agent STRING, property_id STRING,
          view_duration_sec INT, enquiry_flag BOOLEAN, favorite_flag BOOLEAN,
          event_date DATE, _ingested_at TIMESTAMP
        ) USING DELTA PARTITIONED BY (event_date)""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_ipstack_raw)} (
          ip STRING, response_json STRING, http_status INT, api_called_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.bronze_ipstack_errors)} (
          ip STRING, error_message STRING, http_status INT, api_called_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.silver_listings)} (
          property_id STRING, property_type STRING, bedrooms INT, bathrooms INT,
          suburb STRING, postcode STRING, state STRING, price INT, price_range STRING,
          is_luxury BOOLEAN, land_size_sqm INT
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.silver_ip_dim)} (
          ip STRING, country_code STRING, country_name STRING, visitor_region STRING,
          visitor_city STRING, latitude DOUBLE, longitude DOUBLE, timezone_id STRING,
          isp STRING, is_vpn BOOLEAN, enriched_at TIMESTAMP
        ) USING DELTA""")
        spark.sql(f"""CREATE TABLE IF NOT EXISTS {T(self.c.silver_visits_enriched)} (
          event_id STRING, session_id STRING, event_ts TIMESTAMP, user_id STRING, ip STRING,
          property_id STRING, view_duration_sec INT, enquiry_flag BOOLEAN, favorite_flag BOOLEAN,
          event_date DATE, visitor_country STRING, visitor_region STRING, visitor_city STRING,
          visitor_state STRING, timezone_id STRING, device_type STRING,
          suburb STRING, listing_state STRING, property_type STRING, price_range STRING,
          is_luxury BOOLEAN, price INT
        ) USING DELTA PARTITIONED BY (event_date)""")

    def setup(self):
        t0 = time.time()
        self.create_tables()
        print(f"✅ Setup completed in {int(time.time()-t0)}s")

SetupHelper().setup()

